# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 08 · Arrival transfer and synchronized relationships

**Resume Round 3 after the storage-precision guard correction. No new feature round.**

The original notebook stopped during the 32-play smoke because it compared unrounded float64 recomputation with historical float32 storage. The recovery installer verifies the original storage conversion exactly, preserves parent work and the frozen sample, and records source/checkpoint lineage. Model settings, features, targets, and folds are unchanged.

Run cells in order. The verified setup and selection below are reused without package installation or selection scanning. No fabricated NFL results are included. Existing private outputs are preserved in the recovery backup. Save this canonical notebook with Ctrl+S.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round3')
OUT = Path('/home/sagemaker-user/nfl-feature-round3-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Upload/extract the kit in the existing NFL space first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, online=False):
    cmd=[str(PY),str(KIT/'run_round.py'),stage]
    if online: cmd.append('--online')
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout: print(line,end='')
        code=process.wait()
    except KeyboardInterrupt:
        process.send_signal(2); process.wait(timeout=10); raise
    if code:
        raise RuntimeError(f'{stage} stopped with exit {code}. Preserve outputs; inspect the log and export the report. Do not change hyperparameters.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()


In [ ]:
show(visuals.previous_results(KIT),'round2_results')
show(visuals.removals(KIT),'arrival_removals')

## Reuse the verified setup — no installation or old fitting
The recovery receipt must pass before continuing. Your prior preflight, locked runtime self-test and 1,024-play selection have already succeeded. These checks read their receipts; they do not launch setup or fit models.

In [ ]:
recovery = json.loads((OUT/'recovery.json').read_text())
audit = json.loads((OUT/'precision_audit.json').read_text())
plan = json.loads((OUT/'plan.json').read_text())
runtime = json.loads((OUT/'runtime.json').read_text())
assert recovery['status'] == 'precision_patch_applied', 'Run the recovery installer first.'
assert recovery['new_signature'] == plan['signature'], 'Recovery/source identity mismatch.'
assert audit['status'] == 'storage_parity_confirmed', 'Storage parity is not verified.'
assert runtime['status'] == 'tree_runtime_ready', 'Prior runtime success receipt is missing.'
print(json.dumps({k: recovery[k] for k in ['status','prior_feature_checkpoints_preserved','new_model_fits','feature_definition_changes','frozen_selection_unchanged']}, indent=2))
print('Exact stored-precision audit:', audit['plays_verified'], 'plays;', audit['rows_verified'], 'rows')

## Reuse the frozen 1,024-play sample
No new selection or raw-file indexing is performed. The installer preserves the selected game/play list and chronological folds, and records the old and new metadata hashes.

In [ ]:
selection = json.loads((OUT/'selection.json').read_text())
assert selection['signature'] == plan['signature']
assert selection['target_plays'] == 1024 and len(selection['plays']) == 1024
print('Reusing', len(selection['plays']), 'frozen plays. No index rerun.')
show(visuals.sample_balance(OUT),'sample_balance')

## Resume the 32-play feature smoke
This is the previously failed stage. Stored parent X/y arrays are not overwritten. Rebuilt X must match exactly after the original float32 storage conversion. Any mismatch after conversion remains a hard stop. Completed verified feature checkpoints are reused. No scientific model is fitted.

In [ ]:
run('smoke')
show(visuals.sample_history(OUT),'observed_pair_example')

## Prepare the remaining plays; retain completed checkpoints
Original base X/y rows are reused; new pair inputs are built from observed raw data. Labels are read only for added plays and never enter pair construction. The 600-second cap is a stop limit, not a runtime estimate.

In [ ]:
run('prepare')
show(visuals.support(OUT),'pair_support')
print(json.loads((OUT/'preparation.json').read_text()))

**Stop here unless preparation says `expanded_features_ready`.** Save the notebook. Open notebook 09 next. Do not rerun earlier rounds or alter their model parameters.